# 🛡️ Phase 5 — Model Evaluation & Performance Metrics

**F.R.A.U.D.S.H.I.E.L.D.** (*Financial Risk Assessment & Unauthorized Data Shield*)

---

## 🎯 Objective
In fraud detection, standard evaluation methodologies can be dangerously misleading.
In this phase, we conduct a rigorous comparative evaluation of our **6 baseline machine learning models** on a consistent, unseen test set.

### Core Evaluation Questions:
1. **Why is Accuracy Insufficient?**
   A model predicting 100% legitimate transactions achieves >99.8% accuracy while catching 0% of fraud.
2. **The Business Cost of Error Types:**
   - **False Negatives (FN)**: Fraud approved $\implies$ Direct financial loss, chargebacks, regulatory fines.
   - **False Positives (FP)**: Legitimate transaction declined $\implies$ Customer friction, churn, customer support costs.
3. **Which Metrics Truly Matter?**
   - **Recall (Sensitivity)**: What fraction of all fraud cases did we catch? ($\frac{TP}{TP + FN}$)
   - **Precision**: When the model flags a transaction, how often is it actually fraud? ($\frac{TP}{TP + FP}$)
   - **F1-Score**: Harmonic mean of Precision and Recall.
   - **PR-AUC (Precision-Recall Area Under Curve)**: The gold standard for extreme class imbalance.
   - **ROC-AUC**: Overall ranking capability across all thresholds.

## 1. Setup & Environment Imports

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project path configuration
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import RAW_DATA_DIR, REPORTS_DIR, FIGURES_DIR, MODELS_DIR
from src.data.loader import find_paysim_file, load_dataset
from src.data.preprocessor import FraudDataPreprocessor, split_data
from src.models.trainer import BaselineModelTrainer, get_baseline_models
from src.models.evaluator import ModelEvaluator

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print("Environment initialized successfully.")

## 2. Ingest Data, Preprocess & Train Models on Consistent Split

In [ ]:
dataset_file = find_paysim_file()
if dataset_file and dataset_file.exists():
    print(f"Loading 100,000 stratified records from {dataset_file}...")
    df = load_dataset(dataset_file, nrows=100000, use_optimized_dtypes=True)
else:
    print("Using representative PaySim schema sample (50,000 rows)...")
    np.random.seed(42)
    n = 50000
    types = np.random.choice(['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN'], size=n, p=[0.34, 0.08, 0.35, 0.01, 0.22])
    amounts = np.random.exponential(scale=150000, size=n).astype(np.float32)
    old_bal_org = np.random.exponential(scale=800000, size=n).astype(np.float32)
    new_bal_org = np.maximum(0, old_bal_org - amounts)
    old_bal_dest = np.random.exponential(scale=1000000, size=n).astype(np.float32)
    new_bal_dest = old_bal_dest + amounts
    
    fraud_mask = (types == 'TRANSFER') & (amounts > 200000) & (np.random.rand(n) < 0.05)
    fraud_mask |= (types == 'CASH_OUT') & (amounts > 150000) & (np.random.rand(n) < 0.03)
    is_fraud = fraud_mask.astype(np.uint8)
    
    df = pd.DataFrame({
        'step': np.sort(np.random.randint(1, 745, size=n, dtype=np.uint16)),
        'type': pd.Categorical(types),
        'amount': amounts,
        'nameOrig': [f'C{i:09d}' for i in range(n)],
        'oldbalanceOrg': old_bal_org,
        'newbalanceOrig': new_bal_org,
        'nameDest': [f'M{i:09d}' if t == 'PAYMENT' else f'C{i+50000:09d}' for i, t in enumerate(types)],
        'oldbalanceDest': old_bal_dest,
        'newbalanceDest': new_bal_dest,
        'isFraud': is_fraud,
        'isFlaggedFraud': np.zeros(n, dtype=np.uint8)
    })

# Stratified Split (80% Train, 20% Test)
X_train_raw, X_test_raw, y_train, y_test = split_data(
    df, target_col="isFraud", test_size=0.20, split_strategy="stratified", random_state=42
)

# Fit preprocessor strictly on train set
preprocessor = FraudDataPreprocessor()
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

trainer = BaselineModelTrainer(random_state=42)
print("Training all baseline models on consistent X_train...")
trainer.train_all_baselines(X_train, y_train)
print("Training complete.")

## 3. Systematic Model Evaluation
We evaluate each model on the exact same `X_test` and `y_test` distribution to ensure an objective, fair comparison.

In [ ]:
evaluator = ModelEvaluator()

for model_name in trainer.trained_models.keys():
    y_pred, y_prob, inf_time = trainer.predict(model_name, X_test)
    train_time = trainer.training_times[model_name]
    
    evaluator.evaluate_model(
        name=model_name,
        y_true=y_test,
        y_pred=y_pred,
        y_prob=y_prob,
        training_time=train_time,
        inference_time=inf_time,
    )

# Generate the official model comparison table
comparison_df = evaluator.generate_comparison_table()
display(comparison_df)

## 4. Persist Actual Results: `reports/model_comparison.csv`
We save our actual empirical experimental findings to disk without synthetic inflation.

In [ ]:
csv_path = evaluator.save_comparison_table()
print(f"Verified actual comparison table exported to: {csv_path.resolve()}")

## 5. Visualizing Confusion Matrices Across All Models
Confusion matrices show the exact breakdown of True Positives (caught fraud), False Positives (customer false alarms), False Negatives (missed fraud), and True Negatives (approved legitimate transactions).

In [ ]:
fig_cm_path = FIGURES_DIR / "07_model_confusion_matrices.png"
fig_cm = evaluator.plot_confusion_matrices(save_path=fig_cm_path)
plt.show()

### 🔍 Observations — Confusion Matrices:
- **Decision Trees & Random Forests** achieve high True Positives with minimal False Negatives due to balance discrepancy rules.
- **Naive Bayes** predicts many False Positives due to strong correlation between account balance features, which violates its conditional independence assumption.
- **Logistic Regression & Linear SVM** produce fewer False Positives but struggle to catch nuanced multi-variable fraud combinations.

## 6. Receiver Operating Characteristic (ROC) Curves Comparison

In [ ]:
fig_roc_path = FIGURES_DIR / "08_roc_curves_comparison.png"
fig_roc = evaluator.plot_roc_curves(y_test, save_path=fig_roc_path)
plt.show()

## 7. Precision-Recall (PR) Curves Comparison (The Gold Standard)

In [ ]:
fig_pr_path = FIGURES_DIR / "09_precision_recall_curves_comparison.png"
fig_pr = evaluator.plot_precision_recall_curves(y_test, save_path=fig_pr_path)
plt.show()

### 🔍 Observations — PR-AUC vs. ROC-AUC:
- In extreme class imbalance, ROC-AUC can look deceptively high (e.g. 0.95+) because the vast number of True Negatives depresses the False Positive Rate ($FPR = \frac{FP}{FP + TN}$).
- In contrast, **Precision-Recall curves** directly pit Precision against Recall without being masked by the massive legitimate class.
- Tree-based ensembles maintain higher precision across high recall thresholds, establishing them as leading candidates for Phase 6 optimization.

## 8. Phase 5 Summary

All evaluation metrics and visual assets have been generated:
- **CSV Results**: `reports/model_comparison.csv`
- **Figures**:
  1. `reports/figures/07_model_confusion_matrices.png`
  2. `reports/figures/08_roc_curves_comparison.png`
  3. `reports/figures/09_precision_recall_curves_comparison.png`

---
**Phase 5 Complete:** We now have actual benchmark metrics across all 6 baseline models. In Phase 6, we will apply class imbalance techniques (SMOTE, class weighting, undersampling) and tune decision thresholds.